## Getting started with ML project with MLFLOW
- Installing MLflow.
- Starting a local MLflow tracking server.
- Logging and registering a model with MLflow.
- Loading a logged model for inference using MLflow's pyfunc flavor.
- Viewing the experiment results in the MLflow UI.

In [33]:
import sqlite3
from typing import cast

import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import (
    convert_input_example_to_serving_input,
    infer_signature,
    predict,
    validate_serving_input,
)
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils import Bunch

In [2]:
## set the tracking uri
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

In [3]:
## load the dataset
X, y = datasets.load_iris(return_X_y=True)
## split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)

## define the model hyperparameters
params = {"solver": "lbfgs", "max_iter": 10000, "random_state": 8888}

## train the model
lr = LogisticRegression(**params)
lr.fit(X_train,y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",8888
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",10000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use 

In [4]:
X

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
       [4.9, 3

In [5]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [6]:
X_test

array([[6.7, 3.1, 4.7, 1.5],
       [6.8, 3.2, 5.9, 2.3],
       [7.7, 2.6, 6.9, 2.3],
       [5. , 3.4, 1.5, 0.2],
       [5.2, 3.5, 1.5, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [4.9, 3.6, 1.4, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [7.7, 2.8, 6.7, 2. ],
       [7.1, 3. , 5.9, 2.1],
       [5.7, 2.8, 4.1, 1.3],
       [6. , 2.9, 4.5, 1.5],
       [4.7, 3.2, 1.3, 0.2],
       [7.7, 3.8, 6.7, 2.2],
       [4.8, 3. , 1.4, 0.1],
       [6.5, 3.2, 5.1, 2. ],
       [5.7, 4.4, 1.5, 0.4],
       [6.4, 3.1, 5.5, 1.8],
       [5. , 3.3, 1.4, 0.2],
       [6.3, 2.7, 4.9, 1.8],
       [6.6, 3. , 4.4, 1.4],
       [6.1, 3. , 4.9, 1.8],
       [7. , 3.2, 4.7, 1.4],
       [6.3, 2.5, 5. , 1.9],
       [7.7, 3. , 6.1, 2.3],
       [7.3, 2.9, 6.3, 1.8],
       [4.4, 2.9, 1.4, 0.2],
       [6. , 3.4, 4.5, 1.6],
       [6.2, 2.8, 4.8, 1.8],
       [5.6, 2.5, 3.9, 1.1]])

In [7]:
## prediction on the test dataset
y_pred = lr.predict(X_test)
y_pred

array([1, 2, 2, 0, 0, 0, 0, 0, 2, 2, 1, 1, 0, 2, 0, 2, 0, 2, 0, 2, 1, 2,
       1, 2, 2, 2, 0, 1, 1, 1])

In [8]:
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.9666666666666667


In [9]:
# uv run mlflow server --host 127.0.0.1 --port 5000
# uv run mlflow ui

In [10]:
## mlflow tracking - set the tracking uri
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

## create a new mlflow experiment
mlflow.set_experiment("MLFLOW Quickstart")

## Start an MLFLOW run
with mlflow.start_run():
    ## log the hyperparameters
    mlflow.log_params(params)

    ## log the accuracy metrics
    mlflow.log_metric("accuracy", float(accuracy))

    ## set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basis LR model for iris data")

    ## infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    ## log the model
    model_info = mlflow.sklearn.log_model(  # pyright: ignore[reportPrivateImportUsage]    
        sk_model = lr,
        name = "iris-model",
        signature = signature,
        input_example = X_train,
        registered_model_name = "tracking-quickstart",
    )

Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2026/11/09 21:56:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 5
Created version '5' of model 'tracking-quickstart'.


🏃 View run brawny-jay-808 at: http://127.0.0.1:5000/#/experiments/2/runs/9ba3fd97b618497d9becc9648ce4f4f9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [11]:
runs = mlflow.search_runs(experiment_names=["MLFLOW Quickstart"])
runs[["run_id", "metrics.accuracy", "params.solver", "params.max_iter", "start_time"]]

,run_id,metrics.accuracy,params.solver,params.max_iter,start_time
0,9ba3fd97b618497d9becc9648ce4f4f9,0.966667,lbfgs,10000,2026-11-09 20:56:32.138+00:00
1,af1e0f6137ae4b479f1d4f7bace9562d,0.966667,newton-cg,10000,2026-11-07 20:40:13.925+00:00
2,b4757b356c8d4c4ba78d56ca689eb104,0.966667,lbfgs,10000,2026-11-07 20:39:13.216+00:00
3,8014ab7908244ac690082eff0559ae5f,0.933333,newton-cg,10000,2026-11-06 00:06:55.507+00:00
4,9d5a2ab8e3174fb8b79e564a39122af0,0.933333,lbfgs,10000,2026-11-05 23:48:19.246+00:00


In [12]:
con = sqlite3.connect("../mlflow.db")   # path relative to the notebook in 1-MLproject/
con.execute("SELECT run_uuid, timestamp, value, step FROM metrics WHERE key='accuracy'").fetchall()

[('9d5a2ab8e3174fb8b79e564a39122af0', 1793922499326, 0.9333333333333333, 0),
 ('8014ab7908244ac690082eff0559ae5f', 1793923615597, 0.9333333333333333, 0),
 ('b4757b356c8d4c4ba78d56ca689eb104', 1794083953300, 0.9666666666666667, 0),
 ('af1e0f6137ae4b479f1d4f7bace9562d', 1794084014032, 0.9666666666666667, 0),
 ('9ba3fd97b618497d9becc9648ce4f4f9', 1794257792221, 0.9666666666666667, 0)]

#### We are going to do another experiment

In [13]:
## define the model hyperparameters
params = {"solver": "newton-cg", "max_iter": 10000, "random_state": 1000}

## train the model
lr = LogisticRegression(**params)
lr.fit(X_train,y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",1000
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'newton-cg'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",10000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0


In [14]:
## prediction on the test dataset
y_pred = lr.predict(X_test)
y_pred

array([1, 2, 2, 0, 0, 0, 0, 0, 2, 2, 1, 1, 0, 2, 0, 2, 0, 2, 0, 2, 1, 2,
       1, 2, 2, 2, 0, 1, 1, 1])

In [15]:
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.9666666666666667


In [16]:
## Start an MLFLOW run
with mlflow.start_run():
    ## log the hyperparameters
    mlflow.log_params(params)

    ## log the accuracy metrics
    mlflow.log_metric("accuracy", float(accuracy))

    ## set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basis LR model for iris data")

    ## infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    ## log the model
    model_info = mlflow.sklearn.log_model(  # pyright: ignore[reportPrivateImportUsage]    
        sk_model = lr,
        name = "iris-model",
        signature = signature,
        input_example = X_train,
        registered_model_name = "tracking-quickstart",
    )

Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2026/11/09 21:56:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 6
Created version '6' of model 'tracking-quickstart'.


🏃 View run thoughtful-fly-130 at: http://127.0.0.1:5000/#/experiments/2/runs/e82cb6badfdc4a4187e1c7b0095aa919
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


## Inferencing and Validating Model

In [17]:
model_info.model_uri

'models:/m-db1b8570b3ed4201b422af768bcf3efe'

In [18]:
# model_uri = "models:/m-67cb2760f9f24688bc6b4c07c72316b4"   # was 'runs:/<run_id>/iris_model'
model_uri = model_info.model_uri

# The model is logged with an input example. MLFlow converts
# it into the serving payload format for the deployed model endpoint,
# and, in the old MLFlow version, saves it to 'serving_input_payload.json'.
pyfunc_model = mlflow.pyfunc.load_model(model_uri)
serving_payload = convert_input_example_to_serving_input(pyfunc_model.input_example)

if serving_payload is None:
    raise ValueError("Model was logged without an input_example")

# Validate the serving payload works on the model
validate_serving_input(model_uri, serving_payload)

/tmp/ipykernel_36056/1462031831.py:14: FutureWarning: ``mlflow.models.utils.validate_serving_input`` is deprecated since 3.13.0. This method will be removed in a future release. Use ``mlflow.models.predict`` instead.
  validate_serving_input(model_uri, serving_payload)


array([0, 1, 1, 1, 1, 1, 2, 1, 2, 0, 0, 2, 1, 2, 1, 1, 0, 1, 2, 1, 1, 0,
       2, 0, 2, 0, 1, 0, 1, 2, 1, 1, 0, 2, 2, 1, 0, 0, 0, 2, 2, 0, 1, 0,
       1, 2, 2, 0, 0, 0, 0, 1, 1, 2, 1, 1, 0, 0, 2, 0, 0, 2, 0, 2, 0, 0,
       1, 0, 2, 0, 1, 2, 2, 0, 1, 2, 1, 1, 1, 0, 1, 0, 1, 0, 2, 0, 2, 0,
       2, 1, 2, 1, 1, 2, 1, 0, 1, 1, 2, 2, 0, 1, 1, 2, 2, 1, 2, 1, 1, 1,
       2, 0, 2, 0, 0, 0, 2, 1, 0, 2])

In [19]:
# This is my new test data for which we are going to do the prediction.
# This is a list of lists.
pyfunc_model.input_example

array([[4.8, 3. , 1.4, 0.3],
       [5. , 2.3, 3.3, 1. ],
       [5. , 2. , 3.5, 1. ],
       [5.7, 2.8, 4.5, 1.3],
       [5.1, 2.5, 3. , 1.1],
       [5.7, 2.9, 4.2, 1.3],
       [5.7, 2.5, 5. , 2. ],
       [5.5, 2.5, 4. , 1.3],
       [6.5, 3. , 5.5, 1.8],
       [4.7, 3.2, 1.6, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [6.1, 2.6, 5.6, 1.4],
       [5.9, 3. , 4.2, 1.5],
       [5.8, 2.7, 5.1, 1.9],
       [6.1, 3. , 4.6, 1.4],
       [6.5, 2.8, 4.6, 1.5],
       [5. , 3.5, 1.6, 0.6],
       [6.9, 3.1, 4.9, 1.5],
       [6.7, 3. , 5.2, 2.3],
       [6.8, 2.8, 4.8, 1.4],
       [5.8, 2.6, 4. , 1.2],
       [4.4, 3. , 1.3, 0.2],
       [6.7, 2.5, 5.8, 1.8],
       [4.8, 3.1, 1.6, 0.2],
       [6.4, 2.8, 5.6, 2.1],
       [5.1, 3.5, 1.4, 0.3],
       [5.8, 2.7, 4.1, 1. ],
       [5.1, 3.4, 1.5, 0.2],
       [4.9, 2.5, 4.5, 1.7],
       [6.4, 3.2, 5.3, 2.3],
       [5.6, 3. , 4.5, 1.5],
       [5.6, 2.7, 4.2, 1.3],
       [5.5, 3.5, 1.3, 0.2],
       [6. , 2.7, 5.1, 1.6],
       [7.2, 3

In [20]:
serving_payload

'{\n  "inputs": [\n    [\n      4.8,\n      3.0,\n      1.4,\n      0.3\n    ],\n    [\n      5.0,\n      2.3,\n      3.3,\n      1.0\n    ],\n    [\n      5.0,\n      2.0,\n      3.5,\n      1.0\n    ],\n    [\n      5.7,\n      2.8,\n      4.5,\n      1.3\n    ],\n    [\n      5.1,\n      2.5,\n      3.0,\n      1.1\n    ],\n    [\n      5.7,\n      2.9,\n      4.2,\n      1.3\n    ],\n    [\n      5.7,\n      2.5,\n      5.0,\n      2.0\n    ],\n    [\n      5.5,\n      2.5,\n      4.0,\n      1.3\n    ],\n    [\n      6.5,\n      3.0,\n      5.5,\n      1.8\n    ],\n    [\n      4.7,\n      3.2,\n      1.6,\n      0.2\n    ],\n    [\n      4.9,\n      3.0,\n      1.4,\n      0.2\n    ],\n    [\n      6.1,\n      2.6,\n      5.6,\n      1.4\n    ],\n    [\n      5.9,\n      3.0,\n      4.2,\n      1.5\n    ],\n    [\n      5.8,\n      2.7,\n      5.1,\n      1.9\n    ],\n    [\n      6.1,\n      3.0,\n      4.6,\n      1.4\n    ],\n    [\n      6.5,\n      2.8,\n      4.6,\n      1.

In [21]:
type(serving_payload)

str

In [22]:
## A code snippet adpated to the current version of MLFlow

# model_uri = "models:/m-67cb2760f9f24688bc6b4c07c72316b4"   # was 'runs:/<run_id>/iris_model'
model_uri = model_info.model_uri

# The model is logged with an input example. MLFlow converts
# it into the serving payload format for the deployed model endpoint,
# and, in the old MLFlow version, saves it to 'serving_input_payload.json'.
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

# Validate the serving payload works on the model
predict(
    model_uri=model_uri,
    input_data=pyfunc_model.input_example,
    env_manager="uv",   # builds a fresh env from the model's requirements, like a deployment would
)

2026/11/09 21:57:20 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2026/11/09 21:57:20 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/tmp1alal5lx/envs/virtualenv_envs/mlflow-bda841dae3c5a84f2cf4e010f37b1ef6c4b031f1 with python version 3.12.3 using uv
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /tmp/tmp1alal5lx/envs/virtualenv_envs/mlflow-bda841dae3c5a84f2cf4e010f37b1ef6c4b031f1
Activate with: source /tmp/tmp1alal5lx/envs/virtualenv_envs/mlflow-bda841dae3c5a84f2cf4e010f37b1ef6c4b031f1/bin/activate
2026/11/09 21:57:20 INFO mlflow.utils.virtualenv: Installing dependencies
Using Python 3.12.3 environment at: /tmp/tmp1alal5lx/envs/virtualenv_envs/mlflow-bda841dae3c5a84f2cf4e010f37b1ef6c4b031f1
Resolved 4 packages in 217ms
Installed 4 packages in 42ms
 + packaging==26.3
 + pip==26.3
 + setuptools==84.2.1
 + wheel==0.48.0
Using Python 3.12.3 environment at: /tmp/tmp1alal5lx/envs/virtuale

{"predictions": [0, 1, 1, 1, 1, 1, 2, 1, 2, 0, 0, 2, 1, 2, 1, 1, 0, 1, 2, 1, 1, 0, 2, 0, 2, 0, 1, 0, 1, 2, 1, 1, 0, 2, 2, 1, 0, 0, 0, 2, 2, 0, 1, 0, 1, 2, 2, 0, 0, 0, 0, 1, 1, 2, 1, 1, 0, 0, 2, 0, 0, 2, 0, 2, 0, 0, 1, 0, 2, 0, 1, 2, 2, 0, 1, 2, 1, 1, 1, 0, 1, 0, 1, 0, 2, 0, 2, 0, 2, 1, 2, 1, 1, 2, 1, 0, 1, 1, 2, 2, 0, 1, 1, 2, 2, 1, 2, 1, 1, 1, 2, 0, 2, 0, 0, 0, 2, 1, 0, 2]}

In [23]:
## A code snippet adpated to the current version of MLFlow

# model_uri = "models:/m-67cb2760f9f24688bc6b4c07c72316b4"   # was 'runs:/<run_id>/iris_model'
model_uri = model_info.model_uri

# The model is logged with an input example. MLFlow converts
# it into the serving payload format for the deployed model endpoint,
# and, in the old MLFlow version, saves it to 'serving_input_payload.json'.
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

# Validate the serving payload works on the model
predict(
    model_uri=model_uri,
    input_data=pyfunc_model.input_example,
    env_manager="local",   # this runs the prediction in my current environment
)

2026/11/09 21:57:38 INFO mlflow.models.python_api: It is highly recommended to use `uv` as the environment manager for predicting with MLflow models as its performance is significantly better than other environment managers. Run `pip install uv` to install uv. See https://docs.astral.sh/uv/getting-started/installation for other installation methods.
2026/11/09 21:57:38 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


{"predictions": [0, 1, 1, 1, 1, 1, 2, 1, 2, 0, 0, 2, 1, 2, 1, 1, 0, 1, 2, 1, 1, 0, 2, 0, 2, 0, 1, 0, 1, 2, 1, 1, 0, 2, 2, 1, 0, 0, 0, 2, 2, 0, 1, 0, 1, 2, 2, 0, 0, 0, 0, 1, 1, 2, 1, 1, 0, 0, 2, 0, 0, 2, 0, 2, 0, 0, 1, 0, 2, 0, 1, 2, 2, 0, 1, 2, 1, 1, 1, 0, 1, 0, 1, 0, 2, 0, 2, 0, 2, 1, 2, 1, 1, 2, 1, 0, 1, 1, 2, 2, 0, 1, 1, 2, 2, 1, 2, 1, 1, 1, 2, 0, 2, 0, 0, 0, 2, 1, 0, 2]}

## Load the model back for prediction as a generic python function model

In [24]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
predictions = loaded_model.predict(X_test)

iris_features_name = cast(Bunch, datasets.load_iris()).feature_names

result = pd.DataFrame(X_test, columns=iris_features_name)
result["actual_class"] = y_test
result["predicted_class"] = predictions

In [25]:
result

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),actual_class,predicted_class
0,6.7,3.1,4.7,1.5,1,1
1,6.8,3.2,5.9,2.3,2,2
2,7.7,2.6,6.9,2.3,2,2
3,5.0,3.4,1.5,0.2,0,0
4,5.2,3.5,1.5,0.2,0,0
5,5.0,3.4,1.6,0.4,0,0
6,4.9,3.6,1.4,0.1,0,0
7,5.4,3.7,1.5,0.2,0,0
8,7.7,2.8,6.7,2.0,2,2
9,7.1,3.0,5.9,2.1,2,2


In [26]:
result[:5]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),actual_class,predicted_class
0,6.7,3.1,4.7,1.5,1,1
1,6.8,3.2,5.9,2.3,2,2
2,7.7,2.6,6.9,2.3,2,2
3,5.0,3.4,1.5,0.2,0,0
4,5.2,3.5,1.5,0.2,0,0


## Model Registry
The MLFLow Model Registry component is a centralized model store, set of APIs, and UI, to collaboratelly manage the full lifecycle of an MLflow Model. It provides model lineage (which MLflow experiment and run produced the model), model versioning, model aliasing, model tagging, and annotations.

In [30]:
## log the model
# import mlflow.sklearn


# model_info = mlflow.sklearn.log_model(
#     sk_model=lr,
#     artifact_path="iris_model",
#     signature=signature,
#     input_example=X_train,
#     registered_model_name="tracking-quickstart"
# )

"""
Do not the model at the beginning because you dom't know if it is going to be the best model you can achieve or not. Instead follow the way of work described in the following cells.
"""

"\nDo not the model at the beginning because you dom't know if it is going to be the best model you can achieve or not. Instead follow the way of work described in the following cells.\n"

In [34]:
## create a new MLFLOW experiment
mlflow.set_experiment("MLFLOW Quickstart")

## Start an MLFLOW run
with mlflow.start_run():
    ## Log the hyperparameters
    mlflow.log_params(params)

    ## Log the accuracy metrics
    mlflow.log_metric("accuracy", 1.0)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info2", "Basic LR model for iris data")

    ## Infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    ## log the model
    model_info = mlflow.sklearn.log_model(  # pyright: ignore[reportPrivateImportUsage]
        sk_model=lr,
        artifact_path="iris_model",
        signature=signature,
        input_example=X_train,
    )

2026/11/09 22:23:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run merciful-midge-920 at: http://127.0.0.1:5000/#/experiments/2/runs/341da1a4fe59482ba6121d4eeeb88e2a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [40]:
## Inferencing from model from model registry

model_name = "tracking-quickstart"
model_version = "latest"

model_uri = f"models:/{model_name}/{model_version}"

model = mlflow.sklearn.load_model(model_uri = model_uri)    # pyright: ignore[reportPrivateImportUsage]
assert model is not None
model

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",1000
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'newton-cg'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",10000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0


In [41]:
model_uri

'models:/tracking-quickstart/latest'

In [42]:
y_pred_new = model.predict(X_test)  
y_pred_new

array([1, 2, 2, 0, 0, 0, 0, 0, 2, 2, 1, 1, 0, 2, 0, 2, 0, 2, 0, 2, 1, 2,
       1, 2, 2, 2, 0, 1, 1, 1])